# Frequency Shift Keying Project - Part 3: Matched Filter

In [ ]:
## Load numpy and matplotlib
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

from IPython.display import Audio

## Background

A communications system aims to send information from one location to
another. In a digital communications system, the information consists
of a sequence of bits. Simply put, a digital communications system
uses the following steps to accomplish that goal:
* The transmitter is provided a sequence of bits that must be sent
  to the receiver; the receiver is in a different location from the
  transmitter.
* The transmitter maps the sequence of bits into an analog signal
  or waveform.
* The signal is sent to the receiver; this may happen via a cabled
  connection or wirelessly. As the signal travels from transmitter to
  the receiver it becomes weaker and it is disturbed by *noise*.
* The receiver observes this signal and employs knowledge of how
  the transmitter maps bits to signals to reconstruct the original bit
  sequence. 

In Parts 1 and 2, you explored how textual information is mapped to a
signal that is suitable for transmission over either a cable or wirelessly.
This lab focuses on the so-called receiver frontend. The frontend of
the receiver filters the received signal with filters that pass the
transmitted pulses (tones for FSK) and reject signals that do not look
like pulses. These filters are often called matched filters. In the
final lab, you will learn how the bit sequence is recovered from the
matched filter outputs.

### Import Code from Parts 1 and 2

This part of the project builds on the functions that you developed in Parts 1 and 2. 

To make these functions accessible, we need to store them in a *module*. To do so, proceed as follows.

1. Copy the file `fsk.py` that you created for Part 2 into the directory that contains this notebook.
  + this file should already contain
    - an instruction to import Numpy
    - function definitions for functions `int_to_bits()`, `bits_to_int()`, `string_to_bits()`, `bits_to_string()`
2. Copy and paste the code for the two functions from Part 2 into the file:
  * `fsk_modulator()`
  * `fsk_transmitter()`

  To import your code and to verify that the imports worked, execute the following cell: 

In [ ]:
# import the top-level functions
from fsk import string_to_bits, bits_to_string, fsk_transmitter, fsk_modulator

# test bit to/from string conversions
str = "Information"
bits = string_to_bits(str)
assert(str == bits_to_string(bits))

# test FSK signal generation
f0 = 4e3
f1 = 7e3
T = 20e-3
fs = 22050
sig = fsk_transmitter(str, f0, f1, T, fs)

assert(len(sig) == len(bits) * int(T * fs))

print("OK")

## Matched Filter

Matched filters are filters that are used to detect signals of
interest. They assume that the shape of the signals is known but we
may not know when these signals arrive, how strong they are, or - in
our case - which of two or more possible signals was sent.

Principally, matched filters perform an operation that is called
*correlation*. The correlation between two discrete-time signals
$x[n]$ and $y[n]$ at lag $n$ is defined as
$$
  R_{xy}[n] = \sum_k x[n+k] \cdot y[k].
$$

Note that this is very similar to the convolution operation, except
that the correlation uses indices $n+k$ whereas the convolution uses
$n-k$. We will explore below how the convolution (specifically the
NumPy function `np.convolve()`) can be used to compute corrrelations.
Of particular interest to us will be the correlation at lag $n=0$, i.e., 
when the two signals are aligned.

Thus, we will focus on
$$
  R_{xy}[0] = \sum_k x[k] \cdot y[k].
$$
To illustrate the concepts above, consider the following example.


### Example

Consider the two signals $x[n] = \{1, 1, 1, 1\}$ and
$y[n] = \{1, 1, -1, -1\}$. 

We compute the correlations (at $n=0$)
* $R_{xx}[0]$ between $x[n]$ and itself, 
* $R_{yy}[0]$ between $y[n]$ and itself, and 
* $R_{xy}[0]$ between $x[n]$ and $y[n]$ as follows

$$
  R_{xx}[0] = \sum_k x[k]\cdot x[k] = 4
$$

$$
  R_{yy}[0] = \sum_k y[k]\cdot y[k] = 4
$$

$$
  R_{xy}[0] = \sum_k x[k]\cdot y[k] = 0.
$$

This example illustrates that the correlation (at $n=0$) 
* is large when we're
correlating signals with themselves. 
* When different signals are
correlated, then the correlation will be smaller
  - the exact value of the correlation depends on how similar the signals are. 
  
These observations are true in general and they form the basis for using the
matched filter in the receiver.

We will discover that we can compute correlations of an input signal $x[n]$ and a known signal $s[n]$ by passing $x[n]$ through a filter that is matched to $s[n]$.

## Matched Filter Receiver Frontend

Your assignment for this lab is to implement the matched filter
frontend for our FSK communication system. 
Before building the actual
matched filters, this initial part explores how we can use the 
NumPy `convolve` function to compute correlation.

### Correlation via Convolution

Assume we want a matched filter to detect the ramp signal $p[n]$,
where
$$
  p[n] = \begin{cases}
  \frac{n+1}{N} & \text{ for $n=0,2, \ldots, N-1$}\\
  0 & \text{otherwise.}
  \end{cases}
$$

This signal is easily generated in NumPy for a given value of
`N` as
```
p = (1 + np.arange(N)) / N;
```

The matched filter is a filter with an impulse response that is the time-reverse of the signal it is matched to.
For the ramp signal $p[n]$, $h_p[n]$ is the impulse response that is
obtained by time-reversing the pulse $p[n]$, i.e., $h_p[n]=p[N-1-n]$.
Rather than an
up-ramp $h_p[n]$ is a down ramp. 

We can generate the matched filter $h_p[n]$
in NumPy either by direct construction of the down-ramp
```
hp = np.arange(N, 0, -1) / N
```
or, more generally, via the NumPy function `flip()` which reverses the elements of the input signal
```
hp = np.flip(p)
```
This second method can be used to quickly generate a matched filter
for any signal of interest.

### Task: Autocorrelation of ramp signal

For the ramp signal $p[n]$ defined above, find the response of the matched filter using the `np.convolve()`.

* For $N=10$, generate the signal `p` and the matched
  filter impulse response `hp` and plot both using the
  `plt.stem()` function. Use different colors and a legend to distinguish the two.
  - Place this plot in the top-panel of a subplot with two rows
  - Check that they have the expected relationship to each other
* Use the function `np.convolve()`  to convolve `p`
  with `hp`. 
  - This is equivalent to passing the signal $p[n]$
    through a filter with impulse response $h_p[n]$. 
  - Make a stem plot of the output signal $y[n]$ in the bottom panel
  of the subplot.
* Check that the output signal
  - has its maximum value at sample number $n=N-1$ and
  - the value of this maximum is equal to
    $$
      \sum_{n=0}^{N-1} p[n]^2 = \sum_{n=0}^{N-1} \left( \frac{n+1}{N} \right)^2 = \frac{N \cdot (N+1) \cdot (2N+1)}{6 \cdot N^2} = 3.85.
    $$

In [ ]:
# make p and hp
N=10
p = (1+np.arange(N))/N
hp = np.flip(p)
y=np.convolve(p, hp)

# plots
plt.subplot(2, 1, 1)
plt.stem(p, linefmt='b-', markerfmt='bo', label='p[n]')
plt.stem(hp, linefmt='r-', markerfmt='ro', label='hp[n]')
plt.legend()
plt.title('Ramp Signal and Matched Filter')

plt.subplot(2, 1, 2)
plt.stem(y)
plt.title('Matched Filter Output')


plt.show()

The above plots illustrate an important fact about the matched filter
that we want to explore further: </br>

> The output of the matched filter at sample number $n=N-1$ is equal to the 
  correlation between $p[n]$ with itself at lag $n=0$.
  The value of this output is equal to $\sum_{k=0}^{N-1} (p[k])^2$.

To see this,
consider again the signal $p[n]$ of duration $N$ samples,
i.e., non-zero samples span the the range $n=0,1, \ldots, N-1$.

Time-reversal of $p[n]$ yields the matched filter $h_p[n] = p[N-1-n]$
for $n=0,1,\ldots,N-1$.

Recall the definition of the convolution operation. The output $y[n]$ from the convolution
of $p[n]$ and $h_p[n]$ is
$$
  y[n] = p[n] * h_p[n]= \sum_{k=0}^{N-1} p[k] \cdot h_p[n-k]  \quad \text{for
    $n=0,1, \ldots, 2N-1$.} 
$$

Since $h_p[n] = p[N-1-n]$, it follows that $h_p[n-k] = p[N-1-(n-k)]$. Therefore, the above
convolution with the matched filter is equal to 
$$
  \begin{align*}
    y[n] = p[n] * h_p[n] & = \sum_{k=0}^{N-1} p[k] \cdot h_p[n-k]\\
                         & = \sum_{k=0}^{N-1} p[k] \cdot p[N-1-(n-k)]  \quad \text{for
    $n=0,1, \ldots, 2N-1$.}
  \end{align*}
$$

Finally, we consider the output of the matched filter at sample number $n=N-1$. 
Note that for $n=N-1$, $p[N-1-(n-k)] = p[k]$ and therefore:
$$
y[N-1] = \sum_{k=0}^{N-1} (p[k])^2.
$$
This is also equal to the correlation of $p[n]$ with itself at lag $n=0$.

**Conclusions:** We have shown that
1. the correlation of two signals can be computed by convolution with the time-reversed signal,
2. the $(N-1)$-th ouput sample when convolving a length-$N$ signal $p[n]$ with the matched filter for that signal equals $\sum_{k=0}^{N-1} (p[k])^2$

### Matched Filter for FSK Signals

In Part 2 of the lab, we generated an FSK signal by concatenating
tones of frequencies $f_0$ and $f_1$, each of duration $T$, depending
on whether a 0 or a 1 bit is to be sent.

We can interpret the two tones of duration $T$ as our signals of
interest and we want to construct a matched filter for each of the two
tones. 

Throughout the remainder of this lab, we will assume
the FSK parameter values in the cell below.
However, you should parameterize your code so that you can easily
switch to different parameter values.



In [ ]:
# FSK parameters
T = 20e-3
f0 = 1/(2*T) 
f1 = 1/T 
fs = 22050

# number of samples per bit
N = int(fs*T)

#### Task: Construct matched filters for the two tones

To construct the two tones `p0` and `p1` as well as the two matched filters `hp0` and `hp1`, proceed as follows

1. The tone for a 0-bit, can be computed using the `fsk_modulator` function: `p0 = fsk_modulator([0], f0, f1, T, fs)`
2. The tone `p1` for a 1-bit can be computed similarly.
3. The impulse response `h0` of the matched filter for `p0` is obtained by time-reversing `p0` using `np.flip()`.`
4. The impulse response `h1` of the matched filter for `p1` can be computed similarly.

Note that all four signals have length  $N=f_sT$ samples.

In [ ]:
# construct the two tones
p0 = fsk_modulator([0], f0, f1, T, fs)
p1 = fsk_modulator([1], f0, f1, T, fs)

# construct the two matched filters
h0 = np.flip(p0)
h1 = np.flip(p1)

Make a figure with two subplots showing a total of four signals:

1. In the top panel, plot the following two signals using line plots
  + the result of the convolution of pulse `p0` and impulse response `h0`
  + the the result of convolution of pulse `p0` and impulse response `h1`
  + for both signals, mark the sample value at sample number $n=N-1$
2. In the bottom panel, plot the following two signals using line plots
  + the result of the convolution of pulse `p1` and impulse response `h0`
  + the the result of convolution of pulse `p1` and impulse response `h1`
  + for both signals, mark the sample value at sample number $n=N-1$

What are the output vaues from the four filter at sample number $n=N-1$?

In [ ]:
# plots

y00=np.convolve(p0, h0)
y01=np.convolve(p0, h1)

y10=np.convolve(p1, h0)
y11=np.convolve(p1, h1)

plt.subplot(2, 1, 1)
plt.plot(y00, label='p0 through h0')
plt.plot(y01, label='p0 through h1')
plt.plot(N-1, y00[N-1], 'ro')
plt.plot(N-1, y01[N-1], 'bo')
plt.legend()
plt.title('Bit 0 Signal Through Both Filters')

plt.subplot(2, 1, 2)
plt.plot(y10, label='p1 through h0')
plt.plot(y11, label='p1 through h1')
plt.plot(N-1, y10[N-1], 'ro')
plt.plot(N-1, y11[N-1], 'bo')
plt.legend()


plt.show()

# print values of matched filter outputs at n=N-1
print("y00:", y00[N-1])
print("y01:", y01[N-1])
print("y10:", y10[N-1])
print("y11:", y11[N-1])

From your plots, you will see that the matched filters produce:
* a large output value at $n=N-1$ when a signal is passed through its matched filter,
* a small output value at $n=N-1$ when a signal is passed through the filter matched to the other signal.

### Task: Pass FSK Signal through Matched Filters

Finally, we will pass an FSK signal that comprises multiple bits through the matched filters `h0` and `h1`. Proceed as follows:

1. Use the `fsk_transmitter()` function from Part 2, to generate an FSK signal `s` for the string `'E'`.
   * the resulting signal will carry 8 bits
2. Pass the signal `s` through each of the two matched filters:
   * that means, convolve `s` with `h0` to produce the signal `r0`
   * and convolve `s` with `h1` to produce the signal `r1`
3. Plot both signals `r0` and `r1` 
4. Mark the sample values `r0[n]` and `r1[n]` at sample numbers $n=m N - 1$ for each of the 8 bits, i.e., for  $m=1,2,\ldots,8$.
5. Print the values of the eight transmitted bits.


In [ ]:
str = 'E'
s = fsk_transmitter(str, f0, f1, T, fs)

r0= np.convolve(s, h0)
r1 = np.convolve(s, h1)

plt.plot(r0, label='s through h0')
plt.plot(r1, label='s through h1')
for m in range(1, 9):
    n = m*N - 1
    plt.plot(n, r0[n], 'ro')
    plt.plot(n, r1[n], 'bo')
plt.legend()

plt.show()

### Task: Decoding the Matched Filter output 

Describe the relationship between transmitted bits and
matched filter outputs that is evident from this plot. 

Your description should start, for example, with: "In bit periods when a 0
bit is transmitted, the output of the matched filter ...".

**Answer:** in a bit period when the 0 bit is transmitted we get a large magnitude for the matched filter h0 and when we get a large magnitude for the matched filter of h1 we have a 1 bit and h0 is near zero comparing them at the sampling point

## Questions

In addition to performing and documenting the programming exercises above, in your
report also answer the following questions.
Be concise and specific; each question can be answered in 1-2 sentences.

1. Looking at the plot you made in the previous task, why is it important to know the correct samples to check when decoding bits? 
2. In this example, we chose samples $n=mN-1$ with $m=1,2,...8$.
If we were receiving this signal after a wireless transmission, what additional information would we need to know in order to choose the correct samples? 
3. Suppose we decide to speed up transmission by having each period transmit one of 16 frequencies, with each frequency corresponding to one hexadecimal digit. In order to decode the received waveform, how many matched filters will we need and why? 


### Answers:

1. because at those points we get a full bit period of samples so we can correctly decode our bits seeing if our matched filters give a large or small magnitude 

2. We would need to know when the recieved signal starts since it might arrive at sample 100 then 440 would be too soon to sample at

3. we need 16 since now we have 16 frequencies we need one matched filter for each of the 16 frequencies to detect the frequency that corresponds to a hexadecimal digit.

## Summary

This was Part 3 of the Frequency Shift Keying (FSK) project. In this part of the project, we developed code to implement part of an FSK receiver using matched filters to recover the transmitted bit sequence from an FSK signal.

### Deliverable

Submit a PDF version of this notebook. Make sure that:
* All cells are properly typeset and run.
* All code cells are complete (no `FILL_ME_IN` left).
* All plots have proper axis labels and other adornments as appropriate.
* Answers are provided for the final task and the section entitled "Questions."